# Getting the data
The purpose of this notebook is to explore how to access the data from publicly available APIs.

The data access is explained at https://github.com/bundesAPI/smard-api/issues/8 and https://github.com/bundesAPI/smard-api

In [1]:
import time
from datetime import datetime
import requests

FILTER_CODE = "4169" # 4169 is the filter code for wholesale electricity market prices (DE/LU)
REGION = "DE" # DE is the region code for Germany
RESOLUTION = "hour"  # Options: hour, quarterhour, day, week, month, year
BASE_URL = "https://www.smard.de/app/chart_data"

timestamp_url = f"{BASE_URL}/{FILTER_CODE}/{REGION}/index_{RESOLUTION}.json"
timestamp_url

'https://www.smard.de/app/chart_data/4169/DE/index_hour.json'

The timestamp url contains the timestamps and the price_url the prices starting at a certain timestamp. Let's explore the data a bit more

In [2]:
timestamp_response = requests.get(timestamp_url)
timestamps = timestamp_response.json()['timestamps']
timestamps[-1], len(timestamps)

(1785708000000, 410)

In [3]:
early_time = datetime.fromtimestamp(timestamps[0]/1000) #divide by 1000 to turn miliseconds into seconds
new_time = datetime.fromtimestamp(timestamps[-1]/1000) #divide by 1000 to turn miliseconds into seconds
early_time, new_time

(datetime.datetime(2018, 10, 1, 0, 0), datetime.datetime(2026, 8, 3, 0, 0))

# Exploring the data

Now let's see what the price data is for this most recent timestamp

In [4]:
price_url = f"{BASE_URL}/{FILTER_CODE}/{REGION}/{FILTER_CODE}_{REGION}_{RESOLUTION}_{str(timestamps[-1])}.json"
price_response = requests.get(price_url)
price_data = price_response.json()['series']
price_data

[[1785708000000, 139.79],
 [1785711600000, 135.11],
 [1785715200000, 131.94],
 [1785718800000, 129.79],
 [1785722400000, 132.74],
 [1785726000000, 144.3],
 [1785729600000, 169.23],
 [1785733200000, 164.49],
 [1785736800000, 150.0],
 [1785740400000, 135.12],
 [1785744000000, 110.79],
 [1785747600000, 69.68],
 [1785751200000, 26.03],
 [1785754800000, 16.39],
 [1785758400000, 22.56],
 [1785762000000, 60.81],
 [1785765600000, 102.81],
 [1785769200000, 139.8],
 [1785772800000, 169.03],
 [1785776400000, 205.77],
 [1785780000000, 200.67],
 [1785783600000, 185.81],
 [1785787200000, 174.43],
 [1785790800000, 157.14],
 [1785794400000, 154.0],
 [1785798000000, 144.72],
 [1785801600000, 139.0],
 [1785805200000, 137.11],
 [1785808800000, 143.17],
 [1785812400000, 156.95],
 [1785816000000, 170.93],
 [1785819600000, 169.28],
 [1785823200000, 161.49],
 [1785826800000, 146.54],
 [1785830400000, 124.51],
 [1785834000000, 103.74],
 [1785837600000, 87.31],
 [1785841200000, 76.47],
 [1785844800000, 84.06],

In [5]:
for datapoint in price_data:
    print(datetime.fromtimestamp(datapoint[0]/1000))
    print("Electricity costs "+str(datapoint[1])+" euro per MWh")

2026-08-03 00:00:00
Electricity costs 139.79 euro per MWh
2026-08-03 01:00:00
Electricity costs 135.11 euro per MWh
2026-08-03 02:00:00
Electricity costs 131.94 euro per MWh
2026-08-03 03:00:00
Electricity costs 129.79 euro per MWh
2026-08-03 04:00:00
Electricity costs 132.74 euro per MWh
2026-08-03 05:00:00
Electricity costs 144.3 euro per MWh
2026-08-03 06:00:00
Electricity costs 169.23 euro per MWh
2026-08-03 07:00:00
Electricity costs 164.49 euro per MWh
2026-08-03 08:00:00
Electricity costs 150.0 euro per MWh
2026-08-03 09:00:00
Electricity costs 135.12 euro per MWh
2026-08-03 10:00:00
Electricity costs 110.79 euro per MWh
2026-08-03 11:00:00
Electricity costs 69.68 euro per MWh
2026-08-03 12:00:00
Electricity costs 26.03 euro per MWh
2026-08-03 13:00:00
Electricity costs 16.39 euro per MWh
2026-08-03 14:00:00
Electricity costs 22.56 euro per MWh
2026-08-03 15:00:00
Electricity costs 60.81 euro per MWh
2026-08-03 16:00:00
Electricity costs 102.81 euro per MWh
2026-08-03 17:00:00
E

As we can see, some of the prices are still None, we will have to filter for the prices that aren't none 

# Saving it all as a .csv

We will now save all price data locally as a .csv, starting at a starting date parameter

In [6]:
START_DATE = datetime(2022, 1, 1, 0, 0) # January 1st 2022 in order to avoid pandemic effects
start_timestamp = int(START_DATE.timestamp()*1000) #multiply by 1000 to turn seconds into miliseconds
start_timestamp

1640991600000

now concatenate the timestamps list at the start timestamp

In [7]:
import bisect
starttime_index = bisect.bisect_right(timestamps, start_timestamp)
timestamps_concat = timestamps[starttime_index:]
#print the earliest and latest time in the list
early_time = datetime.fromtimestamp(timestamps_concat[0]/1000) #divide by 1000 to turn miliseconds into seconds
new_time = datetime.fromtimestamp(timestamps_concat[-1]/1000) #divide by 1000 to turn miliseconds into seconds
early_time, new_time

(datetime.datetime(2022, 1, 3, 0, 0), datetime.datetime(2026, 8, 3, 0, 0))

In [1]:
import csv, sys


In [18]:

outfile = open("price_data.csv", "w", encoding="utf8")
writer = csv.writer(outfile)

for index , this_timestamp in enumerate(timestamps_concat):
    this_data_url=f"{BASE_URL}/{FILTER_CODE}/{REGION}/{FILTER_CODE}_{REGION}_{RESOLUTION}_{str(this_timestamp)}.json"
    this_price_data = requests.get(this_data_url)
    this_price_datalist =this_price_data.json()['series']
    writer.writerows(this_price_datalist)
    print("Saving price data component "+str(index+1)+"/"+str(len(timestamps_concat)), end="\r", flush=True)
outfile.close()

Saving price data component 240/240

now all the price data since the start date is saved in 'price_data.csv'